In [ ]:
# ============================================================
# Cluster-size ablation on Colon
# Fix all best params, vary only GO-LR k
# ============================================================

import os, sys, gc, random, warnings
warnings.filterwarnings("ignore")

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from gotabpfn import GraphFeatureOrdering, pidf_segpca, TabPFN25Head, TabPFN25Config
PIDFSegPCA = pidf_segpca

# -----------------------
# Config
# -----------------------
SEED = 42
DATA_FILE = "coloncancer_encoded.csv"
TARGET_COL = "label"

# vary only this
K_VALUES = [3, 4, 5, 7, 10, 12, 15]

# fixed best params from Optuna
GO_METRIC = "euclidean"
GO_REFINE_PASSES = 3
GO_DIRECTION_SELECT = True

NSC_SEG = "equal_mass"
NSC_M_RULE = "idf"
NSC_TAU = 0.99
NSC_GAMMA = 1.7570143129240916
NSC_BETA = 0.2244046472232107
NSC_MMIN = 64
NSC_MMAX = 384
NSC_LMIN = 16
ASSUME_STANDARDIZED = False

TABPFN_SEED = 42

# -----------------------
# Utils
# -----------------------
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
        except Exception:
            pass
        torch.cuda.empty_cache()

def ensure_multiclass_contiguous(y: np.ndarray):
    y = np.asarray(y).reshape(-1)
    uniq = np.unique(y)
    uniq_sorted = np.sort(uniq)
    class_map = {orig: i for i, orig in enumerate(uniq_sorted.tolist())}
    y_enc = np.vectorize(class_map.get, otypes=[np.int64])(y).astype(np.int64)
    return y_enc, class_map, int(len(class_map))

def compute_deltas_adjacent_corr(X_tr: np.ndarray, Pi_star: list[int], eps: float = 1e-12) -> torch.Tensor:
    X = torch.from_numpy(X_tr).float()
    perm = torch.tensor(Pi_star, dtype=torch.long)
    Xp = X[:, perm]
    Xc = Xp - Xp.mean(dim=0, keepdim=True)
    std = Xc.std(dim=0, unbiased=False, keepdim=True).clamp_min(eps)
    Z = Xc / std
    corr = (Z[:, :-1] * Z[:, 1:]).mean(dim=0)
    return (1.0 - corr.abs()).cpu()

# -----------------------
# Load data
# -----------------------
seed_everything(SEED)

df = pd.read_csv(DATA_FILE)
y_raw = df[TARGET_COL].to_numpy()
X_df = df.drop(columns=[TARGET_COL])

y_all, class_map, NUM_CLASSES = ensure_multiclass_contiguous(y_raw)

# match your earlier setup: standardize globally
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df.values).astype(np.float32, copy=False)

X_all = np.asarray(X_scaled, dtype=np.float32, order="C")
y_all = np.asarray(y_all, dtype=np.int64)

print(f"[DATA] {DATA_FILE} | X={X_all.shape} | C={NUM_CLASSES} | map={class_map}")

rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)

NSC_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TABPFN_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# fixed TabPFN head config
head_cfg = TabPFN25Config(
    task_type="binary",
    num_classes=int(NUM_CLASSES),
    device=TABPFN_DEVICE,
    random_state=int(TABPFN_SEED),
)

# -----------------------
# Run ablation
# -----------------------
rows = []

for k in K_VALUES:
    print(f"\n===== Evaluating k = {k} =====")
    seed_everything(SEED)

    go = GraphFeatureOrdering(
        num_clusters=int(k),
        metric=GO_METRIC,
        refine=True,
        direction_select=bool(GO_DIRECTION_SELECT),
        refine_passes=int(GO_REFINE_PASSES),
    )

    # fit GO-LR once on full X for this k
    try:
        Pi_star, _, _, _ = go.fit(X_all, seed=SEED, deterministic=True, use_cpu_kmeans=False)
    except RuntimeError:
        cleanup_cuda()
        Pi_star, _, _, _ = go.fit(X_all, seed=SEED, deterministic=True, use_cpu_kmeans=True)

    accs = []

    for fold_id, (tr_idx, va_idx) in enumerate(rkf.split(X_all, y_all), start=1):
        X_tr = X_all[tr_idx]
        y_tr = y_all[tr_idx]
        X_va = X_all[va_idx]
        y_va = y_all[va_idx]

        nsc = PIDFSegPCA(
            segmentation=NSC_SEG,
            l_min=int(NSC_LMIN),
            m_rule=NSC_M_RULE,
            gamma=float(NSC_GAMMA),
            beta=float(NSC_BETA),
            tau=float(NSC_TAU),
            M_min=int(NSC_MMIN),
            M_max=int(NSC_MMAX),
            assume_standardized=bool(ASSUME_STANDARDIZED),
            device=NSC_DEVICE,
        )

        deltas = None if NSC_SEG == "uniform" else compute_deltas_adjacent_corr(X_tr, Pi_star)

        X_tr_t = torch.from_numpy(X_tr)
        nsc.configure(Pi_star=Pi_star, X_train=X_tr_t, tau=float(NSC_TAU), deltas=deltas)

        Z_tr = nsc.compress(X_tr_t, mode="flatten").cpu().numpy()
        Z_va = nsc.compress(torch.from_numpy(X_va), mode="flatten").cpu().numpy()

        head = TabPFN25Head(head_cfg)
        head.fit(Z_tr, y_tr)

        P = head.predict_proba(Z_va)
        pred = np.argmax(P, axis=1)
        acc = float(accuracy_score(y_va, pred))
        accs.append(acc)

        print(f"k={k:2d} | fold {fold_id:02d} | acc={acc:.4f}")
        cleanup_cuda()

    mean_acc = float(np.mean(accs))
    std_acc = float(np.std(accs, ddof=1))

    rows.append({
        "k": k,
        "mean_acc": 100.0 * mean_acc,
        "std_acc": 100.0 * std_acc,
    })

    print(f"--> k={k}: {100.0*mean_acc:.2f} ± {100.0*std_acc:.2f}")

# final table
ablation_df = pd.DataFrame(rows).sort_values("k")
print("\n=== Cluster-size Ablation Results ===")
print(ablation_df.to_string(index=False))

ablation_df["acc_pm"] = ablation_df.apply(
    lambda r: f"{r['mean_acc']:.2f}±{r['std_acc']:.2f}", axis=1
)
print("\n=== LaTeX/Markdown friendly ===")
print(ablation_df[["k", "acc_pm"]].to_string(index=False))

[DATA] coloncancer_encoded.csv | X=(62, 2000) | C=2 | map={0: 0, 1: 1}

===== Evaluating k = 3 =====
k= 3 | fold 01 | acc=0.8462
k= 3 | fold 02 | acc=0.8462
k= 3 | fold 03 | acc=0.6667
k= 3 | fold 04 | acc=0.9167
k= 3 | fold 05 | acc=0.9167
k= 3 | fold 06 | acc=0.9231
k= 3 | fold 07 | acc=0.6923
k= 3 | fold 08 | acc=0.9167
k= 3 | fold 09 | acc=0.8333
k= 3 | fold 10 | acc=0.8333
k= 3 | fold 11 | acc=0.7692
k= 3 | fold 12 | acc=0.7692
k= 3 | fold 13 | acc=1.0000
k= 3 | fold 14 | acc=0.7500
k= 3 | fold 15 | acc=0.9167
k= 3 | fold 16 | acc=0.8462
k= 3 | fold 17 | acc=0.6923
k= 3 | fold 18 | acc=1.0000
k= 3 | fold 19 | acc=0.9167
k= 3 | fold 20 | acc=0.5833
k= 3 | fold 21 | acc=0.7692
k= 3 | fold 22 | acc=0.6923
k= 3 | fold 23 | acc=0.8333
k= 3 | fold 24 | acc=0.8333
k= 3 | fold 25 | acc=0.9167
--> k=3: 82.72 ± 10.69

===== Evaluating k = 4 =====
k= 4 | fold 01 | acc=0.8462
k= 4 | fold 02 | acc=0.8462
k= 4 | fold 03 | acc=0.7500
k= 4 | fold 04 | acc=0.8333
k= 4 | fold 05 | acc=0.9167
k= 4 |